### 14. Encoder 모니터링

#### 1) 모듈 추가

In [1]:
import libraries.Omniwheel_Protocol as Omniwheel_Protocol
import serial
import time

#### 2) 프로토콜 정의

In [2]:
Arduino_ID = 0x20
REQUEST_ODOMETER = 0xA0
ANSWER_ODOMETER = 0xB0
ENCODER_CLEAR = 0xC2

MID_ENCODER_WHEEL_1 = 0x80
MID_ENCODER_WHEEL_2 = 0x81
MID_ENCODER_WHEEL_3 = 0x82

MID_list=[MID_ENCODER_WHEEL_1,
          MID_ENCODER_WHEEL_2,     
          MID_ENCODER_WHEEL_3]


#### 3) 변수 초기화

In [3]:
data_wheel_1_encoder_count=0
data_wheel_2_encoder_count=0
data_wheel_3_encoder_count=0

#### 4) 송수신 객체 생성 및 초기화

In [4]:
send_packet = Omniwheel_Protocol.Packet()
recv_packet = Omniwheel_Protocol.Packet()

In [5]:
recv_packet.clearPacket()
send_packet.clearPacket()

#### 5) 변수 선언

In [6]:
recv_list = []
recv_parsing_packet = []

In [7]:
send_flag=False

#### 6) 통신 포트 정보 초기화

In [8]:
Serial_Arduino = serial.Serial(port ="/dev/ttyACM0" ,baudrate = 115200, timeout=.1)
time.sleep(1)
print("connect complete")

SerialException: [Errno 13] could not open port /dev/ttyTCU0: [Errno 13] Permission denied: '/dev/ttyTCU0'

#### 7) 송신 함수(센서 값 요청)

In [ ]:
def Packet_send(_id, _cmd, _mid, _data = None):
    # 전역변수 사용
    global send_flag

    # 전송 완료 플래그 확인 
    if(send_flag==False):

        # 초기화
        send_packet.clearPacket()

        # ID 설정
        send_packet.setID(_id)

        # CMD 설정
        send_packet.setCMD(_cmd)

        # Payload 초기화
        send_packet.clearPayload()

        # MID, data 설정
        send_packet.addPayload(_mid, _data)

        # 패킷 LRC 계산
        send_packet.calcLRC_Lower()

        # 패킷을 리스트로 변환
        send_list=send_packet.packetToList()

        # 아두이노 에 패킷 리스트 전송
        Serial_Arduino.write(send_list)
        
        # 전송 완료 플래그 설정
        send_flag=True  

#### 8) 수신 함수(데이터 수신)

In [ ]:
def Packet_receive(ser):
    # 전역변수 사용
    global send_flag
    
    # 전송 완료 플래그 확인
    if(send_flag==True):
        
        # 수신받은 데이터가 없을때까지
        while ser.inWaiting() > 0:
            
            # 1바이트씩 데이터를 받음
            Arduino_Data = ser.read(1)
            
            # 데이터를 받은 경우
            if(len(Arduino_Data)>0):
                
                # 수신 패킷 리스트에 수신 데이터 저장
                recv_list.append(ord(Arduino_Data))
                
                # 패킷 종료 데이터를 받은 경우
                if(ord(Arduino_Data)==0x03):
                    
                    # 수신 받은 데이터 파싱
                    if(recv_packet.parsingList(recv_list)):
                        
                        # 파싱한 데이터를 파싱완료 리스트에 저장
                        recv_parsing_packet.append(recv_packet)
                        
                        # 패킷 리스트 초기화
                        recv_list.clear()
                        
                        # 전송 완료 플래그 해제
                        send_flag=False
                        
                        # 반복문 탈출
                        break

#### 9) 수신 리스트 초기화 함수

In [ ]:
def Received_packet():
    # 파싱 완료 리스트의 첫번째 패킷을 result 변수에 저장함
    result=recv_parsing_packet[0]
    
    # 저장 완료한 파싱 완료 리스트 삭제 
    del recv_parsing_packet[0]
    
    # result 변수값 반환
    return result

#### 10) 센서 데이터 저장 함수

In [ ]:
# 인자값으로 Received_packet() 함수에서 반환된 값을 넣어줌
def Encoder_Data(packet):
    # 전역변수를 사용 설정
    global MID_list
    global data_wheel_1_encoder_count
    global data_wheel_2_encoder_count
    global data_wheel_3_encoder_count

    # 패킷에서 ID 추출
    packet_id=packet.getID()
    
    # 패킷에서 CMD 추출
    packet_cmd=packet.getCMD()
    
    # 추출한 ID 가 Arduino_ID 와 맞는지 확인
    if(packet_id==Arduino_ID):

        #추출한 CMD가 응답 CMD가 맞는지 확인
        if(packet_cmd==ANSWER_ODOMETER):

            # 패킷에서 Payload 값을 추출함
            for payload in packet.getPayload():

                # 추출한 MID가 Encoder 센서가 맞는지 확인
                if(payload.getID()==MID_list[0]):
                    
                    # Payload 에서 읽어온 데이터를 저장 
                    data_wheel_1_encoder_count = int(float(payload.getData()))

                elif(payload.getID()==MID_list[1]):
                    # Payload 에서 읽어온 데이터를 저장 
                    data_wheel_2_encoder_count = int(float(payload.getData()))
                    
                elif(payload.getID()==MID_list[2]):
                    # Payload 에서 읽어온 데이터를 저장 
                    data_wheel_3_encoder_count = int(float(payload.getData()))

#### 11) 센서 값 모니터링

In [ ]:
#Encoder 센서 초기화 패킷 송신
Packet_send(Arduino_ID, ENCODER_CLEAR, MID_ENCODER_WHEEL_1)

#전송 완료 플래그  해제
send_flag=False

while(True):
    # MID_list리스트에서 하나씩 mid를 가져옴
    for mid in MID_list:
        
        #데이터 요청 패킷 송신
        Packet_send(Arduino_ID, REQUEST_ODOMETER, mid)
        
        #일정 시간 대기
        time.sleep(0.1)
        
        # 응답 데이터 패킷 수신
        Packet_receive(Serial_Arduino)
        
        # 응답받은 데이터가 있을경우
        if(len(recv_parsing_packet)>0):

                # 파싱완료 리스트에서 첫번째 패킷을 가져옴
                p=Received_packet()

                # 수신한 패킷을 파싱하고 Encoder 센서 측정 값을 저장
                Encoder_Data(p)
            
    # 데이터 출력
    print("X : "+str(data_wheel_1_encoder_count)+" Y: "+str(data_wheel_2_encoder_count)+" Z : "+str(data_wheel_3_encoder_count))
                 
    # 일정시간 대기
    time.sleep(0.5)